<a href="https://colab.research.google.com/github/yrarjun59/COMFYUI/blob/main/Testing_Lora_with_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### TESTING LORA MODELS with COMFYUI

### Cell 1 – Install ComfyUI and dependencies

In [ ]:
# ==========================================
# 1. INSTALL COMFYUI + PYTORCH (CUDA 12.4)
# ==========================================

import os
import torch

# Install PyTorch with CUDA if not already present
if torch.cuda.is_available():
    print(f"✅ PyTorch {torch.__version__} with CUDA already installed.")
else:
    print("📦 Installing PyTorch with CUDA 12.4...")
    !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# Clone ComfyUI if not present
if not os.path.exists("/content/ComfyUI"):
    !git clone -q https://github.com/comfyanonymous/ComfyUI
    print("✅ ComfyUI cloned.")
else:
    print("✅ ComfyUI already exists.")

# Install requirements (skip torch to avoid conflict)
!pip install -q -r /content/ComfyUI/requirements.txt

# Install aria2 (optional but handy)
!apt-get install -y aria2 > /dev/null 2>&1
print("✅ Setup complete.")

### Cell 2 b – Install Custom Nodes (Optional, but useful)

In [ ]:
# ==========================================
# 2. INSTALL CUSTOM NODES (OPTIONAL)
# ==========================================
# These are popular, but you can comment them out if not needed.

nodes = {
    "comfyui-manager": "https://github.com/ltdrdata/ComfyUI-Manager.git",
    "rgthree-comfy": "https://github.com/rgthree/rgthree-comfy.git",
    "ComfyUI-Easy-Use": "https://github.com/yolain/ComfyUI-Easy-Use.git",
}

base = "/content/ComfyUI/custom_nodes"
for name, url in nodes.items():
    path = os.path.join(base, name)
    if not os.path.exists(path):
        print(f"📥 Cloning {name}...")
        !git clone -q {url} {path}
    else:
        print(f"✅ {name} already exists.")

print("✅ Custom nodes installed.")

### Cell 2 -a  – Mount Google Drive

In [ ]:
# ==========================================
# 2. MOUNT GOOGLE DRIVE
# ==========================================

from google.colab import drive
drive.mount('/content/drive')

### Cell 3 – Copy your base model and LoRA from Drive to ComfyUI

In [ ]:
# ==========================================
# 3. COPY MODELS FROM DRIVE TO COMFYUI
# ==========================================

import os
import shutil

# --- Define your Drive file paths ---
# Change these to the actual locations of your files

MODEL_SRC = "/content/drive/MyDrive/Loras/models/RealVisXL_V5.0_fp16.safetensors"
LORA_SRC  = "/content/drive/MyDrive/path/Loras/divya_influencer/output/divya_influencer-10.safetensors"

# --- ComfyUI destinations ---
MODEL_DST = "/content/ComfyUI/models/checkpoints/RealVisXL_V5.0_fp16.safetensors"
LORA_DST  = "/content/ComfyUI/models/loras/divya_influencer-10.safetensors"

# Create directories if they don't exist
os.makedirs(os.path.dirname(MODEL_DST), exist_ok=True)
os.makedirs(os.path.dirname(LORA_DST), exist_ok=True)

# Copy model
if os.path.exists(MODEL_SRC):
    if not os.path.exists(MODEL_DST):
        shutil.copyfile(MODEL_SRC, MODEL_DST)
        print(f"✅ Model copied to {MODEL_DST}")
    else:
        print(f"⏭️ Model already exists at {MODEL_DST}, skipping.")
else:
    print(f"❌ Model not found at {MODEL_SRC}. Please check the path.")

# Copy LoRA
if os.path.exists(LORA_SRC):
    if not os.path.exists(LORA_DST):
        shutil.copyfile(LORA_SRC, LORA_DST)
        print(f"✅ LoRA copied to {LORA_DST}")
    else:
        print(f"⏭️ LoRA already exists at {LORA_DST}, skipping.")
else:
    print(f"❌ LoRA not found at {LORA_SRC}. Please check the path.")

### Cell 4 – (Optional) Copy SDXL VAE if you have it

In [ ]:
# ==========================================
# DOWNLOAD SDXL VAE (FP16 FIX)
# ==========================================
import os

vae_url = "https://huggingface.co/madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl_vae.safetensors"
vae_dst = "/content/ComfyUI/models/vae/sdxl_vae.safetensors"

os.makedirs(os.path.dirname(vae_dst), exist_ok=True)

if not os.path.exists(vae_dst):
    print("📥 Downloading SDXL VAE (320 MB)...")
    !wget -q --nc --show-progress -O "{vae_dst}" "{vae_url}"
    print("✅ VAE downloaded.")
else:
    print("⏭️ VAE already exists, skipping.")

### Cell 5 – Launch ComfyUI with ngrok

In [ ]:
# ==========================================
# 4. LAUNCH COMFYUI WITH NGROK
# ==========================================

!pip install -q pyngrok

from pyngrok import ngrok
import subprocess
import socket
import time
from google.colab import userdata
from datetime import datetime
import IPython.display as display

# ------------------------------------------
# Get NGROK token (set in Colab secrets)
# ------------------------------------------

NGROK_TOKEN = userdata.get("NGROK_TOKEN")

if not NGROK_TOKEN:
    raise ValueError("Please set NGROK_TOKEN in Colab secrets.")

!ngrok config add-authtoken $NGROK_TOKEN

# ------------------------------------------
# Record launch time
# ------------------------------------------

start_time = datetime.now()
print(f"🚀 ComfyUI launch initiated at: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")

# ------------------------------------------
# Start ComfyUI
# ------------------------------------------

print("▶️ Starting ComfyUI...")
comfy_process = subprocess.Popen([
    "python",
    "/content/ComfyUI/main.py",
    "--dont-print-server"
])

# ------------------------------------------
# Wait for port 8188
# ------------------------------------------

port = 8188
print("⏳ Waiting for ComfyUI server...")

while True:
    try:
        sock = socket.create_connection(("127.0.0.1", port), timeout=2)
        sock.close()
        print("\n✅ ComfyUI server is running on port", port)
        break
    except OSError:
        elapsed = datetime.now() - start_time
        print(f"\r⏳ Waiting... {str(elapsed).split('.')[0]} elapsed", end="", flush=True)
        time.sleep(2)

# ------------------------------------------
# Create ngrok tunnel
# ------------------------------------------

print("\n🌐 Creating ngrok tunnel...")
public_url = ngrok.connect(port, bind_tls=True)

print("\n" + "=" * 55)
print("🌐 COMFYUI PUBLIC URL")
print("=" * 55)
print(public_url)
print("=" * 55)

# ------------------------------------------
# Live status monitor
# ------------------------------------------

print("\n🟢 ComfyUI + ngrok are running")
print("🔄 Live uptime monitor started")
print("🛑 Interrupt the cell to stop monitoring\n")

try:
    while True:
        now = datetime.now()
        uptime = now - start_time
        display.clear_output(wait=True)
        print("=" * 60)
        print("🚀 COMFYUI + NGROK LIVE STATUS")
        print("=" * 60)
        print(f"📅 Started: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"🕐 Current: {now.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"⏱️ Uptime: {str(uptime).split('.')[0]}")
        print(f"🔌 ComfyUI Port: {port}")
        print(f"🌐 Public URL: {public_url}")
        print("🟢 Status: RUNNING")
        print("=" * 60)
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Monitor stopped.")